In [1]:
import biom
import pandas as pd
import numpy as np
import biom_to_csv as btc
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
XSB_150bp   = btc.biom_to_csv_phylum('biom_data/emp_deblur_150bp.release1.biom')

In [3]:
def complexity(Xsb, n_iter=1000, tol=1e-10):
    # Salva gli indici e le colonne originali se Xsb è un DataFrame Pandas
    index_s = Xsb.index if isinstance(Xsb, pd.DataFrame) else None
    columns_p = Xsb.columns if isinstance(Xsb, pd.DataFrame) else None

    # Converti in array numpy per il calcolo veloce
    X = np.asarray(Xsb, dtype=np.float64)
    Ns, Np = X.shape
    
    logNs = np.log(Ns)
    logNp = np.log(Np)

    # Entropie iniziali
    Hp_raw, Hs_raw = btc.entropy(Xsb)
    
    Hp = np.squeeze(np.asarray(Hp_raw, dtype=np.float64))
    Hs = np.squeeze(np.asarray(Hs_raw, dtype=np.float64))

    eps = 1e-12
    
    for _ in range(n_iter):
        f = logNs - Hs
        g = logNp - Hp

        # Force dfp and dfc to be non-negative
        dfp = np.maximum(0, X * g[None, :])
        dfc = np.maximum(0, X * f[:, None])

        row_sum = dfp.sum(axis=1, keepdims=True)
        col_sum = dfc.sum(axis=0, keepdims=True)
        
        xi0 = np.divide(dfp, row_sum, out=np.zeros_like(dfp), where=row_sum != 0)
        xi1 = np.divide(dfc, col_sum, out=np.zeros_like(dfc), where=col_sum != 0)

        Hp_prev = Hp.copy()
        Hs_prev = Hs.copy()

        p0_log = np.zeros_like(xi0)
        p1_log = np.zeros_like(xi1)

        mask0 = xi0 > eps
        mask1 = xi1 > eps

        p0_log[mask0] = xi0[mask0] * np.log(xi0[mask0])
        p1_log[mask1] = xi1[mask1] * np.log(xi1[mask1])

        Hs = -np.sum(p0_log, axis=1)
        Hp = -np.sum(p1_log, axis=0)

        delta = np.sqrt(np.sum((Hs - Hs_prev) ** 2) + np.sum((Hp - Hp_prev) ** 2))

        if np.isnan(delta) or delta < tol:
            break

    # Ricostruisci le Series di Pandas mantenendo gli indici/colonne originali
    if columns_p is not None:
        Hp = pd.Series(Hp, index=columns_p, name="Hp")
    if index_s is not None:
        Hs = pd.Series(Hs, index=index_s, name="Hs")

    return Hp, Hs

In [4]:
Hp, Hs = complexity(XSB_150bp)

In [5]:
Hp_0, Hs_0 = btc.entropy(XSB_150bp)

In [6]:
Hp_sorted = Hp.sort_values(ascending=False)
Hs_sorted = Hs.sort_values(ascending=False)

In [7]:
Hp_sorted0 = Hp_0.sort_values(ascending=False)
Hs_sorted0 = Hs_0.sort_values(ascending=False)

In [8]:
Hp_sorted

p__Proteobacteria    9.072040
p__Bacteroidetes     8.997569
p__Actinobacteria    8.599095
p__Planctomycetes    8.343208
p__Firmicutes        8.248293
                       ...   
p__OP1               2.961138
p__NPL-UPA2          2.219987
p__Aquificae         1.267731
p__OctSpA1-106       1.012307
p__Thermotogae      -0.000000
Name: Hp, Length: 81, dtype: float64

In [9]:
Hp_sorted0

p__Proteobacteria    9.070891
p__Bacteroidetes     8.996929
p__Actinobacteria    8.599364
p__Planctomycetes    8.342840
p__Firmicutes        8.248840
                       ...   
p__OP1               2.939414
p__NPL-UPA2          2.217548
p__Aquificae         1.254900
p__OctSpA1-106       1.012473
p__Thermotogae      -0.000000
Length: 81, dtype: float64

In [10]:
Hs_sorted

1288.FBH10Sept07.McMahon.Pool.1.and.1percentPhiX.110310.HWUSI.EAS552R.0357.s.1.1.sequence    1.257736
678.OA.mesocosm.422                                                                          1.210236
678.OA.mesocosm.439                                                                          1.178926
945.P6.G12.lane2.NoIndex.L002                                                                1.164534
945.P4.C11.lane2.NoIndex.L002                                                                1.153333
                                                                                               ...   
550.L5S111.s.5.sequence                                                                     -0.000000
1883.2006.188.Crump.Artic.LTREB.main.lane1.NoIndex                                          -0.000000
1721.S16M                                                                                   -0.000000
1747.c.v.v.2011007.fecal                                                          

In [11]:
Hs_sorted0

678.seasonal.insitu.sample.571                        2.695035
678.seasonal.insitu.sample.595                        2.632708
678.seasonal.insitu.sample.522                        2.622083
678.seasonal.insitu.sample.509                        2.602030
678.seasonal.insitu.sample.596                        2.597338
                                                        ...   
1721.BMC.8.5T.2.1                                    -0.000000
1064.W.CV251                                         -0.000000
1883.2003.062.Crump.Artic.LTREB.main.lane1.NoIndex   -0.000000
1747.lima.991488.saliva                              -0.000000
1064.G.CV298                                         -0.000000
Length: 17483, dtype: float64